# Lab 4 Parte 1 — CNN para detección de grietas en hormigón

**Objetivo:** entrenar una red neuronal convolucional (CNN) que clasifique fotos de superficie de hormigón como con grieta (`Positive`) o sin grieta (`Negative`), y usarla como ejemplo de triaje automático de inspección visual.

**Recorrido de este notebook:**
1. Contexto del dataset y panorama de CNNs en inspección estructural
2. Exploración de datos (EDA): balance de clases, tamaños, ejemplos visuales
3. Transformaciones y *data augmentation*
4. `DataLoader`s de entrenamiento y validación
5. Arquitectura de la CNN
6. Entrenamiento
7. Curvas de entrenamiento (pérdida y accuracy)
8. Métricas de validación (matriz de confusión, reporte de clasificación)
9. Casos locales — inspeccionar predicciones individuales
10. Grad-CAM — qué región de la imagen usó la CNN para decidir
11. Reflexión — preguntas que los alumnos necesitarían

**Entorno:** ejecuta primero `bash labs/setup.sh` (crea `labs/.venv` con PyTorch CPU) y corre este notebook con ese kernel.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import (
    Resize, ToTensor, Normalize, RandomHorizontalFlip, RandomRotation, ColorJitter,
)
from pathlib import Path
from PIL import Image

%matplotlib inline
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Entorno listo | device={device}")

## Contexto del dataset (Mendeley — grietas en hormigón)

| Clase | En obra significa… | Imágenes (subset) |
|-------|-------------------|-------------------|
| **Negative** | Superficie sin grieta visible | 800 train + 200 val |
| **Positive** | Superficie con grieta | 800 train + 200 val |

Imágenes **227×227** RGB; en el lab se redimensionan con `IMAGE_SIZE` para entrenar en CPU. Detalle: [`data/DATOS.md`](data/DATOS.md).

## 1. Panorama CNN en inspección estructural

Una CNN aprende **filtros espaciales** (bordes, texturas, orientación de grietas) directamente de los píxeles, en lugar de que un ingeniero diseñe a mano descriptores como "contraste local" o "densidad de bordes". Cada bloque de **convolución** detecta un patrón local; cada bloque de **pooling** reduce la dimensión y aporta invariancia a pequeños desplazamientos de la grieta dentro del encuadre. Con suficientes ejemplos variados (distinta luz, textura de superficie, ángulo de cámara) el modelo aprende a generalizar a fotos que no ha visto — sin miles de imágenes representativas, en cambio, memoriza detalles del set de entrenamiento y falla en obra.

In [ ]:
COMPONENTES_CNN = ["convolución", "pooling", "ReLU", "flatten", "fully connected"]
print("Componentes CNN del laboratorio:")
for c in COMPONENTES_CNN:
    print(f"  · {c}")

## 2. Exploración de datos (EDA)

Antes de entrenar conviene confirmar tres cosas: que las clases estén balanceadas en train y val, que las imágenes tengan un tamaño consistente, y cómo lucen visualmente las dos clases. Si el dataset estuviera desbalanceado el modelo tendería a predecir la clase mayoritaria; si los tamaños variaran mucho, el resize introduciría distorsión desigual.

In [ ]:
# --- Rutas y conteos ---
RUTA_DATOS = Path("data/cracks_subset")
if not RUTA_DATOS.is_dir():
    raise FileNotFoundError("Ejecuta primero: python tools/prepare_data_lab4_part1.py o bash labs/setup.sh")

conteos = {}
for split in ("train", "val"):
    conteos[split] = {}
    for cls in ("Negative", "Positive"):
        d = RUTA_DATOS / split / cls
        conteos[split][cls] = len(list(d.glob("*.jpg")))

class_names = ["Negative", "Positive"]
display(pd.DataFrame(conteos))

In [ ]:
N_EJEMPLOS_MOSAICO = 4
N_MUESTRAS_EDA = 100

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
x = np.arange(len(class_names))
width = 0.35
axes[0].bar(x - width / 2, [conteos['train'][c] for c in class_names], width, label='train')
axes[0].bar(x + width / 2, [conteos['val'][c] for c in class_names], width, label='val')
axes[0].set_xticks(x, class_names)
axes[0].set_title('Balance por clase'); axes[0].legend()

paths_eda = []
for cls in class_names:
    paths_eda.extend(sorted((RUTA_DATOS / 'train' / cls).glob('*.jpg')))
paths_eda = paths_eda[:N_MUESTRAS_EDA]
widths, heights = [], []
for p in paths_eda:
    with Image.open(p) as im:
        widths.append(im.width)
        heights.append(im.height)
axes[1].hist(widths, bins=15, alpha=0.7, label='ancho')
axes[1].hist(heights, bins=15, alpha=0.7, label='alto')
axes[1].set_title('Tamaños de imagen (muestra train)'); axes[1].legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, N_EJEMPLOS_MOSAICO, figsize=(2 * N_EJEMPLOS_MOSAICO, 4))
for row, cls in enumerate(["Negative", "Positive"]):
    paths = sorted((RUTA_DATOS / 'train' / cls).glob('*.jpg'))[:N_EJEMPLOS_MOSAICO]
    for col, p in enumerate(paths):
        axes[row, col].imshow(Image.open(p))
        axes[row, col].set_title(cls if col == 0 else '')
        axes[row, col].axis('off')
plt.suptitle('Muestras de entrenamiento')
plt.tight_layout()
plt.show()

## 3. Transformaciones y *data augmentation*

En **train** añadimos variación artificial (flip horizontal, rotación, cambios de color) para que el modelo no memorice la posición exacta de cada grieta y generalice mejor. En **val** solo aplicamos resize y normalización — sin augmentation — para que las métricas de validación sean deterministas y comparables entre corridas. Rotar demasiado tiene un límite: una grieta rotada 90° sigue siendo grieta, pero rotaciones extremas o distorsiones agresivas pueden generar imágenes que ya no se parecen a una foto real de obra. La normalización con media/desviación 0.5 centra los píxeles en `[-1, 1]`, lo que ayuda a que el optimizador converja más rápido.

In [ ]:
IMAGE_SIZE = 128
AUG_ROTATION = 15
N_AUG_MOSTRADOS = 4

train_transform = transforms.Compose([
    RandomHorizontalFlip(),
    RandomRotation(AUG_ROTATION),
    ColorJitter(brightness=0.2, contrast=0.2),
    Resize((IMAGE_SIZE, IMAGE_SIZE)),
    ToTensor(),
    Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
val_transform = transforms.Compose([
    Resize((IMAGE_SIZE, IMAGE_SIZE)),
    ToTensor(),
    Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

sample_path = sorted((RUTA_DATOS / 'train' / 'Positive').glob('*.jpg'))[0]
img_pil = Image.open(sample_path).convert('RGB')
fig, axes = plt.subplots(1, N_AUG_MOSTRADOS + 1, figsize=(2.2 * (N_AUG_MOSTRADOS + 1), 2.5))
axes[0].imshow(img_pil)
axes[0].set_title('original')
axes[0].axis('off')
for i in range(N_AUG_MOSTRADOS):
    aug = train_transform(img_pil)
    vis = aug.numpy().transpose(1, 2, 0) * 0.5 + 0.5
    axes[i + 1].imshow(vis.clip(0, 1))
    axes[i + 1].set_title(f'aug {i + 1}')
    axes[i + 1].axis('off')
plt.suptitle('Data augmentation en train')
plt.tight_layout()
plt.show()

## 4. `DataLoader`s

`shuffle=True` solo en train evita que el modelo aprenda el orden en que están guardados los archivos; en val el orden no importa porque solo medimos, así que lo dejamos fijo (`shuffle=False`) para poder comparar corridas. Cada batch tiene forma `N × C × H × W` — N imágenes, 3 canales de color, alto y ancho iguales a `IMAGE_SIZE`.

In [ ]:
BATCH_SIZE = 32
train_ds = datasets.ImageFolder(RUTA_DATOS / 'train', transform=train_transform)
val_ds = datasets.ImageFolder(RUTA_DATOS / 'val', transform=val_transform)
class_names = train_ds.classes
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f"Clases: {class_names} | train={len(train_ds)} | val={len(val_ds)}")

## 5. Arquitectura CNN

Dos bloques `Conv2d + ReLU + MaxPool2d`: el primero aprende bordes y texturas simples, el segundo combina esos patrones en features más abstractas (p. ej. "línea oscura irregular sobre fondo uniforme"). `AdaptiveAvgPool2d` deja el tamaño espacial fijo sin importar la resolución de entrada, y el clasificador final incluye `Dropout` antes de la capa de salida para reducir sobreajuste — apaga neuronas al azar durante el entrenamiento para que el modelo no dependa de una sola combinación de features.

In [ ]:
N_FILTERS = 32
DROPOUT = 0.3

class CrackCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, N_FILTERS, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(N_FILTERS, N_FILTERS * 2, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(N_FILTERS * 2 * 16, 64),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

modelo = CrackCNN().to(device)
print(modelo)

## 6. Entrenamiento

Para clasificación con dos clases usamos `CrossEntropyLoss` (equivalente a softmax + log-verosimilitud negativa) y `Adam` como optimizador — buena combinación por defecto que no requiere afinar mucho la tasa de aprendizaje. Para detectar sobreajuste comparamos la curva de `train_loss` contra `val_loss`: si la de entrenamiento sigue bajando pero la de validación empieza a subir, el modelo está memorizando en vez de generalizar.

In [ ]:
# --- Funciones de entrenamiento y evaluación ---
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total

print("✅ Funciones train_one_epoch / eval_epoch listas.")

In [ ]:
N_EPOCHS = 5
LEARNING_RATE = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(modelo.parameters(), lr=LEARNING_RATE)
history = {k: [] for k in ['train_loss', 'val_loss', 'train_acc', 'val_acc']}
for epoch in range(N_EPOCHS):
    tl, ta = train_one_epoch(modelo, train_loader, criterion, optimizer, device)
    vl, va = eval_epoch(modelo, val_loader, criterion, device)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['train_acc'].append(ta)
    history['val_acc'].append(va)
    print(f"Época {epoch+1}/{N_EPOCHS} | loss {tl:.3f}/{vl:.3f} | acc {ta:.3f}/{va:.3f}")
print("✅ Entrenamiento completado.")

## 7. Curvas de entrenamiento

Si `val_loss` sube mientras `train_loss` sigue bajando, es la señal clásica de sobreajuste — la respuesta suele ser *early stopping* (detener antes) o más regularización (más dropout, más augmentation, o simplemente más datos). En producción, el número de épocas "suficiente" se decide observando dónde `val_loss` deja de mejorar, no un número fijo.

In [ ]:
epochs = range(1, N_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(epochs, history['train_loss'], label='train')
ax1.plot(epochs, history['val_loss'], label='val')
ax1.set_title('Pérdida'); ax1.legend()
ax2.plot(epochs, history['train_acc'], label='train')
ax2.plot(epochs, history['val_acc'], label='val')
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout()
plt.show()

## 8. Métricas en validación

La matriz de confusión y el reporte de clasificación muestran algo que el accuracy global esconde: cuál clase se confunde más, y en qué dirección. En inspección estructural, un **falso negativo** (una grieta real clasificada como `Negative`) es más grave que un falso positivo — un falso positivo genera una revisión de más, un falso negativo puede dejar pasar un daño real. Por eso conviene mirar el *recall* de la clase `Positive`, no solo el accuracy general.

In [ ]:
modelo.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        preds = modelo(images).argmax(1).cpu().numpy()
        y_pred.extend(preds.tolist())
        y_true.extend(labels.numpy().tolist())
acc_val = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicho'); ax.set_ylabel('Real'); ax.set_title('Matriz de confusión (val)')
plt.tight_layout()
plt.show()
print(classification_report(y_true, y_pred, target_names=class_names))
print(f"Accuracy validación: {acc_val:.3f}")

## 9. Casos locales

Mirar predicciones caso por caso — no solo métricas agregadas — ayuda a auditar si la CNN realmente está mirando la grieta o si se está guiando por artefactos como sombras, manchas de humedad o reflejos. Es la manera más directa de detectar cuándo el modelo "acierta por la razón equivocada".

In [ ]:
N_CASOS_MOSTRADOS = 3
modelo.eval()
fig, axes = plt.subplots(1, N_CASOS_MOSTRADOS, figsize=(3 * N_CASOS_MOSTRADOS, 3))
if N_CASOS_MOSTRADOS == 1:
    axes = [axes]
for i in range(N_CASOS_MOSTRADOS):
    img, label = val_ds[i]
    with torch.no_grad():
        pred = modelo(img.unsqueeze(0).to(device)).argmax(1).item()
    vis = img.numpy().transpose(1, 2, 0) * 0.5 + 0.5
    axes[i].imshow(vis.clip(0, 1))
    axes[i].set_title(f"real: {class_names[label]} | pred: {class_names[pred]}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 10. Grad-CAM — ¿en qué región se fijó la CNN?

Una CNN puede acertar por la razón equivocada: por ejemplo, aprender que "esquina más oscura de la foto" correlaciona con `Positive` en el dataset de entrenamiento, sin que eso generalice a fotos nuevas. **Grad-CAM** (*Gradient-weighted Class Activation Mapping*) responde a la pregunta "¿qué píxeles empujaron esta predicción?": toma el gradiente de la clase predicha respecto a los mapas de activación de la última capa convolucional, los usa como pesos de importancia, y produce un mapa de calor sobre la imagen original.

Esto es exactamente el tipo de evidencia que un inspector de campo necesitaría antes de confiar en una alerta automática: si el mapa de calor se concentra sobre la grieta real, la predicción es defendible; si se concentra en una esquina, una sombra o el borde de la foto, la predicción es sospechosa aunque el modelo tenga alta confianza — y esa distinción no es visible con solo mirar el accuracy o la matriz de confusión.

In [ ]:
class GradCAM:
    """Grad-CAM sobre la última capa convolucional de CrackCNN (self.features[3])."""

    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_activations)
        target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradients(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, x, class_idx=None):
        self.model.zero_grad()
        logits = self.model(x)
        if class_idx is None:
            class_idx = logits.argmax(1).item()
        logits[0, class_idx].backward()

        # Pesos = promedio global de los gradientes por canal (GAP sobre H, W)
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx, logits.softmax(dim=1)[0, class_idx].item()


# self.features[3] es el segundo Conv2d (el último bloque convolucional antes del pooling adaptativo)
grad_cam = GradCAM(modelo, modelo.features[3])

# Buscamos casos con ambas clases reales, priorizando al menos un error de clasificación si existe
modelo.eval()
candidatos = []
for i in range(len(val_ds)):
    img, label = val_ds[i]
    with torch.no_grad():
        pred = modelo(img.unsqueeze(0).to(device)).argmax(1).item()
    candidatos.append((i, label, pred))

errores = [c for c in candidatos if c[1] != c[2]]
aciertos_por_clase = {0: next((c for c in candidatos if c[1] == 0 and c[1] == c[2]), None),
                       1: next((c for c in candidatos if c[1] == 1 and c[1] == c[2]), None)}
muestra_gradcam = [c for c in [aciertos_por_clase[0], aciertos_por_clase[1]] if c]
muestra_gradcam += errores[:2]
muestra_gradcam = muestra_gradcam[:6] if len(muestra_gradcam) >= 4 else candidatos[:6]

fig, axes = plt.subplots(1, len(muestra_gradcam), figsize=(3 * len(muestra_gradcam), 3.2))
if len(muestra_gradcam) == 1:
    axes = [axes]
for ax, (idx, label, pred) in zip(axes, muestra_gradcam):
    img, _ = val_ds[idx]
    x = img.unsqueeze(0).to(device)
    cam, class_idx, prob = grad_cam(x, class_idx=pred)
    vis = img.numpy().transpose(1, 2, 0) * 0.5 + 0.5
    ax.imshow(vis.clip(0, 1))
    ax.imshow(cam, cmap='jet', alpha=0.45)
    correcto = "✅" if label == pred else "⚠️"
    ax.set_title(f"{correcto} real:{class_names[label]} pred:{class_names[pred]} ({prob:.0%})", fontsize=9)
    ax.axis('off')
plt.suptitle('Grad-CAM — región que impulsó la predicción')
plt.tight_layout()
plt.show()

¿Confiarías en esta alerta? Un inspector de campo miraría el mapa de calor exactamente así: si el rojo cae sobre la grieta visible, la predicción es explicable y defendible en un informe; si cae sobre una esquina oscura o un reflejo, la alerta necesita revisión humana antes de tomarse como evidencia — la CNN puede tener 95% de confianza y aun así estar mirando el lugar equivocado de la imagen.

## Reflexión: preguntas que los alumnos necesitarían

- ¿Dónde desplegarías este modelo en la práctica — inspección de dron, cámaras fijas en un puente, foto de un técnico en campo? ¿Qué cambia en cada caso?
- Si el Grad-CAM de una predicción "Positive" se concentra fuera de cualquier grieta visible, ¿qué harías antes de reportar esa alerta a un ingeniero responsable?
- El dataset tiene clases balanceadas (800/800). ¿Qué pasaría con las métricas y con el comportamiento del modelo si en un edificio real solo el 2% de las fotos tuviera grietas?
- ¿Un falso negativo (grieta real clasificada como `Negative`) y un falso positivo tienen el mismo costo en este contexto? ¿Cómo cambiarías el umbral de decisión si no fuera así?
- ¿Esta CNN sustituye la inspección normativa de un perito estructural, o qué rol específico cumple dentro de un flujo de inspección real (triaje, priorización, documentación)?